In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'


metadata_df['sample_title'] = metadata_df['sample_title'].astype(str).str.strip()
metadata_df['sample_id'] = metadata_df['sample_title'].str.extract(r'([\d\-]+)')
metadata_df['sample_id'] = metadata_df['sample_id'].astype(str).str.strip()

metadata_df

,sample_title,disease status,sample_id
samples,,,
GSM1464282,B08-3483NA,healthy,08-3483
GSM1464283,B08-3758NA,healthy,08-3758
GSM1464284,B08-4386NA,healthy,08-4386
GSM1464285,B08-4579NA,healthy,08-4579
GSM1464286,N08-4497A,healthy,08-4497
...,...,...,...
GSM1464363,T10-5656A,small cell lung cancer,10-5656
GSM1464364,T11-4067A,small cell lung cancer,11-4067
GSM1464365,T11-548A,small cell lung cancer,11-548


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)

gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)


df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()
# df_combined.to_csv(f'{GEO_ID}_raw_counts_NCBI.csv', index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_10334/868395266.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,disease status,sample_id,DDX11L1,WASH7P,MIR6859-1,MIR1302-2HG,MIR1302-2,FAM138A,OR4F5,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM1464282,B08-3483NA,healthy,08-3483,0,102,2,0,0,1,1,...,43385,281,375,308,42630,16172,1675,12686,22,1166
GSM1464283,B08-3758NA,healthy,08-3758,9,105,1,4,0,4,2,...,97598,115,137,210,81617,34833,2155,31413,51,494
GSM1464284,B08-4386NA,healthy,08-4386,2,110,8,0,1,1,0,...,23704,249,316,365,27303,10187,1191,9155,62,761
GSM1464285,B08-4579NA,healthy,08-4579,2,81,1,5,3,0,3,...,17512,203,486,1194,27073,16225,1279,8638,61,237
GSM1464286,N08-4497A,healthy,08-4497,24,204,0,0,0,6,0,...,44906,122,246,681,44954,17419,1171,17776,53,651
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM1464363,T10-5656A,small cell lung cancer,10-5656,7,67,1,0,0,0,0,...,59092,279,321,212,32943,14643,1992,12972,16,861
GSM1464364,T11-4067A,small cell lung cancer,11-4067,1,104,0,0,0,0,0,...,48929,777,852,686,56933,27483,3837,15562,102,2185
GSM1464365,T11-548A,small cell lung cancer,11-548,25,270,16,0,0,9,4,...,27765,319,530,916,35974,21338,2224,12836,71,494


In [3]:
# Download supplementary data from: https://pmc.ncbi.nlm.nih.gov/articles/instance/4836692/bin/pgen.1005895.s012.xlsx

df_clinical_data = pd.read_excel('pgen.1005895.s012.xlsx', sheet_name=0, skiprows=1)
df_clinical_data = df_clinical_data.iloc[:-1, :]
df_clinical_data.columns.values[0] = 'sample_id'
df_clinical_data['sample_id'] = df_clinical_data['sample_id'].astype(str)

# we include patients who had successful RNA sequencing data and clinical data
df_clinical_data = df_clinical_data[
    (df_clinical_data['Tumor RNAseq data'].notna() | df_clinical_data['Normal RNA seq'].notna()) & 
    (df_clinical_data['Months'].notna()) &
    (df_clinical_data['Survival Status'].notna())
]

df_clinical_data.to_csv('pgen.1005895.s012.csv', index=False)

# df_clinical_data
df_combined = df_clinical_data.merge(
    df_combined,
    left_on='sample_id',
    right_on='sample_id',
    how='inner'
)

data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined = df_combined[df_combined['disease status'] != 'healthy']
df_combined.to_csv(data_file_name, index=False)
df_combined

,sample_id,Tumor exome data,Normal exome data a,Tumor RNAseq data,Normal RNA seq,Race,Dx Date,Survival Date,Age,Gender,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
0,5,X,NaN,X,NaN,Chinese,2004-02-23,2009-05-10,54.43,2.0,...,206750,547,902,1407,123181,45101,7154,79820,120,3307
1,6,X,NaN,X,NaN,Chinese,2004-03-10,2004-11-17,66.65,2.0,...,253885,82,64,91,123744,39882,4093,91915,53,1963
2,15,X,X,X,NaN,Chinese,2005-08-29,2007-07-10,70.17,1.0,...,8181,36,49,58,6372,2554,399,3558,3,158
3,18,X,X,X,NaN,Chinese,2006-05-08,2007-07-19,50.03,1.0,...,5473,46,66,92,11860,9054,1196,2488,26,30
4,20,X,X,X,NaN,Chinese,2006-10-07,2008-08-29,65.42,1.0,...,30825,73,74,79,26715,7709,921,12174,11,277
5,21,X,X,X,NaN,Chinese,2006-12-21,2009-04-20,50.77,1.0,...,10840,102,103,122,12568,5655,911,4506,7,253
6,25,X,X,X,NaN,Chinese,2007-07-24,2009-04-20,46.69,1.0,...,34620,57,78,61,20272,7842,748,9687,4,130
7,26,X,X,X,NaN,Chinese,2007-09-27,2008-03-24,44.34,2.0,...,12330,49,53,74,8997,3358,363,5058,4,96
8,27,X,X,X,NaN,Chinese,2007-10-08,2009-02-28,39.44,1.0,...,33530,54,65,53,18682,4736,622,12190,5,272
9,29,X,X,X,NaN,Chinese,2009-03-25,2010-03-25,64.36,1.0,...,35912,84,76,52,22475,6250,705,13695,11,312


# Generate description


In [4]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, 'pgen.1005895.s012.csv')
    print("Description generated")

desc_data['data_summary'] = {
    'samples': df_clinical_data.shape[0],
    'clinical_features': df_combined.shape[1] - df_raw_counts.shape[1],
    'genes': df_raw_counts.shape[1]
}

print("Description updated")
    
desc_data['data_file_names'] = [data_file_name]
with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated
